# 🎙️ DubbingStation - Google Colab GPU Voice Worker (T4 / A100)
Sổ tay khởi chạy **Hybrid Voice Worker** trên GPU Google Colab miễn phí.
Hỗ trợ: **Coqui XTTS-v2** zero-shot voice cloning, **Piper VITS (VIVOS Corpus)** và đường hầm **Cloudflare Tunnel** công khai kết nối trực tiếp về DubbingStation.

In [ ]:
# Bước 1: Kiểm tra GPU và cài đặt thư viện cần thiết
!nvidia-smi
!pip install -q coqui-tts soundfile fastapi uvicorn pydantic python-multipart

In [ ]:
# Bước 2: Tạo tệp voice_worker.py độc lập trên Colab
%%writefile voice_worker.py
import os
import sys
import io
import time
import base64
import tempfile
import re
from typing import List, Optional, Dict, Any
from pathlib import Path

os.environ['COQUI_TOS_AGREED'] = '1'

import uvicorn
from fastapi import FastAPI, HTTPException, Body
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel

app = FastAPI(
    title="DubbingStation Voice Worker",
    version="3.0.0",
    description="Hybrid Voice Synthesis & Cloning Worker for Vietnamese and Multilingual Speech"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

NUMBERS_MAP = {
    '0': 'không', '1': 'một', '2': 'hai', '3': 'ba', '4': 'bốn',
    '5': 'năm', '6': 'sáu', '7': 'bảy', '8': 'tám', '9': 'chín'
}

def number_to_words(num_str: str) -> str:
    try:
        val = int(num_str)
        if val == 0:
            return 'không'
        if len(num_str) <= 4:
            words = [NUMBERS_MAP.get(ch, ch) for ch in num_str]
            return ' '.join(words)
        return num_str
    except Exception:
        return num_str

def normalize_vietnamese_text(text: str) -> str:
    if not text:
        return ""
    t = text.replace('%', ' phần trăm ').replace('$', ' đô la ').replace('₫', ' đồng ').replace('&', ' và ').replace('+', ' cộng ')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    def replace_num(match):
        num = match.group(0)
        if len(num) <= 4:
            return f" {number_to_words(num)} "
        return num
    t = re.sub(r'\b\d+\b', replace_num, t)
    return re.sub(r'\s+', ' ', t).strip()

cached_latents: Dict[str, Any] = {}
loaded_models: Dict[str, Any] = {"xtts": None}

def get_device() -> str:
    import torch
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

def load_xtts_engine():
    global loaded_models
    if loaded_models["xtts"] is not None:
        return loaded_models["xtts"]
    try:
        from TTS.api import TTS
        device_str = get_device()
        use_gpu = device_str == "cuda"
        model_name = os.getenv("XTTS_MODEL", "tts_models/multilingual/multi-dataset/xtts_v2")
        print(f"[Worker] Đang nạp mô hình Coqui XTTS ({model_name}) trên {device_str.upper()}...")
        loaded_models["xtts"] = TTS(model_name=model_name, progress_bar=False, gpu=use_gpu)
        print("[Worker] Nạp Coqui XTTS thành công!")
        return loaded_models["xtts"]
    except Exception as e:
        print(f"[Worker Warning] Không thể nạp Coqui XTTS: {e}")
        return None

class TrainVoiceRequest(BaseModel):
    voice_id: str
    audio_paths: Optional[List[str]] = None
    audio_base64_list: Optional[List[str]] = None
    gender: Optional[str] = "neutral"
    language: Optional[str] = "vi-VN"

class SynthesizeVoiceRequest(BaseModel):
    voice_id: Optional[str] = None
    text: str
    speaker_wavs: Optional[List[str]] = None
    speaker_base64: Optional[List[str]] = None
    language: Optional[str] = "vi"
    speed: Optional[float] = 1.0
    temperature: Optional[float] = 0.75
    repetition_penalty: Optional[float] = 5.0
    preferred_engine: Optional[str] = "auto"

@app.get("/health")
def health():
    import torch
    device = get_device()
    gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU Host"
    vram_mb = round(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024)) if device == "cuda" else 0
    return {
        "status": "ONLINE",
        "worker_name": "DubbingStation-Colab-Voice-Worker",
        "version": "3.0.0",
        "device": device,
        "is_gpu": device == "cuda",
        "gpu_name": gpu_name,
        "vram_mb": vram_mb,
        "xtts_ready": loaded_models["xtts"] is not None or os.getenv("XTTS_ENABLED", "true") == "true",
        "cached_voices_count": len(cached_latents),
        "cached_voices": list(cached_latents.keys()),
        "supported_datasets": ["VIVOS (AILAB)", "VietTTS", "OpenSLR 57"],
    }

@app.post("/train")
def train_voice(req: TrainVoiceRequest):
    tts = load_xtts_engine()
    if not tts:
        raise HTTPException(status_code=503, detail="XTTS Engine chưa sẵn sàng trên worker.")
    xtts_model = getattr(tts.synthesizer, "tts_model", None)
    if not xtts_model:
        raise HTTPException(status_code=500, detail="Không thể truy cập mô hình XTTS synthesizer.")
    temp_files = []
    resolved_paths = []
    try:
        if req.audio_paths:
            for p in req.audio_paths:
                if os.path.exists(p):
                    resolved_paths.append(p)
        if req.audio_base64_list:
            for idx, b64_str in enumerate(req.audio_base64_list):
                if not b64_str:
                    continue
                clean_b64 = b64_str.split(",")[-1]
                data = base64.b64decode(clean_b64)
                tmp = tempfile.NamedTemporaryFile(suffix=f"_train_{idx}.wav", delete=False)
                tmp.write(data)
                tmp.close()
                temp_files.append(tmp.name)
                resolved_paths.append(tmp.name)
        if not resolved_paths:
            raise HTTPException(status_code=400, detail="Không tìm thấy tệp âm thanh hợp lệ.")
        print(f"[Worker] Đang trích xuất conditioning latents cho {req.voice_id} từ {len(resolved_paths)} mẫu...")
        gpt_cond_latent, speaker_embedding = xtts_model.get_conditioning_latents(
            audio_path=resolved_paths,
            max_ref_length=60,
            gpt_cond_len=30,
            sound_norm_refs=True,
        )
        cached_latents[req.voice_id] = {
            "gpt_cond_latent": gpt_cond_latent,
            "speaker_embedding": speaker_embedding,
            "gender": req.gender,
            "updated_at": time.time(),
        }
        return {
            "success": True,
            "voice_id": req.voice_id,
            "samples_processed": len(resolved_paths),
            "cached_in_memory": True,
            "message": f"Đã trích xuất và tối ưu hóa latents cho giọng {req.voice_id} thành công!",
        }
    except Exception as e:
        print(f"[Worker Error] Trích xuất latents thất bại: {e}")
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        for tmp_path in temp_files:
            try:
                os.remove(tmp_path)
            except Exception:
                pass

@app.post("/synthesize")
def synthesize(req: SynthesizeVoiceRequest):
    clean_text = normalize_vietnamese_text(req.text)
    if not clean_text:
        raise HTTPException(status_code=400, detail="Văn bản cần đọc không hợp lệ hoặc trống.")
    temp_files = []
    try:
        tts = load_xtts_engine()
        if not tts:
            raise HTTPException(status_code=503, detail="XTTS Engine chưa sẵn sàng.")
        xtts_model = getattr(tts.synthesizer, "tts_model", None)
        device = get_device()
        gpt_cond_latent = None
        speaker_embedding = None

        if req.voice_id and req.voice_id in cached_latents:
            lat = cached_latents[req.voice_id]
            gpt_cond_latent = lat["gpt_cond_latent"]
            speaker_embedding = lat["speaker_embedding"]
            if hasattr(gpt_cond_latent, "to"):
                gpt_cond_latent = gpt_cond_latent.to(device)
            if hasattr(speaker_embedding, "to"):
                speaker_embedding = speaker_embedding.to(device)
        else:
            resolved_refs = []
            if req.speaker_wavs:
                resolved_refs.extend([p for p in req.speaker_wavs if os.path.exists(p)])
            if req.speaker_base64:
                for idx, b64 in enumerate(req.speaker_base64):
                    clean_b64 = b64.split(",")[-1]
                    data = base64.b64decode(clean_b64)
                    tmp = tempfile.NamedTemporaryFile(suffix=f"_synth_ref_{idx}.wav", delete=False)
                    tmp.write(data)
                    tmp.close()
                    temp_files.append(tmp.name)
                    resolved_refs.append(tmp.name)
            if resolved_refs:
                gpt_cond_latent, speaker_embedding = xtts_model.get_conditioning_latents(
                    audio_path=resolved_refs,
                    max_ref_length=60,
                    gpt_cond_len=30,
                    sound_norm_refs=True,
                )

        if gpt_cond_latent is not None and speaker_embedding is not None:
            req_lang = (req.language or "vi").lower().split("-")[0]
            supported_langs = getattr(tts, "languages", []) or []
            use_lang = req_lang if req_lang in supported_langs else os.getenv("XTTS_FALLBACK_LANG", "en")
            out = xtts_model.inference(
                text=clean_text,
                language=use_lang,
                gpt_cond_latent=gpt_cond_latent,
                speaker_embedding=speaker_embedding,
                temperature=req.temperature or 0.75,
                repetition_penalty=req.repetition_penalty or 5.0,
                speed=req.speed or 1.0,
            )
            import soundfile as sf
            buf = io.BytesIO()
            sf.write(buf, out["wav"], 24000, format="WAV", subtype="PCM_16")
            buf.seek(0)
            return Response(
                content=buf.read(),
                media_type="audio/wav",
                headers={"X-Engine": "Coqui-XTTS-v2", "X-Device": device}
            )
        raise HTTPException(status_code=400, detail="Không tìm thấy speaker embedding hợp lệ.")
    finally:
        for tmp_p in temp_files:
            try:
                os.remove(tmp_p)
            except Exception:
                pass

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8020)

In [ ]:
# Bước 3: Khởi động Worker và tạo đường hầm Cloudflare Tunnel công khai
import os, subprocess, time

print("[1/3] Cài đặt Cloudflare Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("[2/3] Đang khởi động FastAPI Worker trên GPU...")
worker_proc = subprocess.Popen(["python", "-m", "uvicorn", "voice_worker:app", "--host", "0.0.0.0", "--port", "8020"])
time.sleep(3)

print("[3/3] Đang mở kết nối Cloudflare...")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8020"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel_proc.stdout:
    if "trycloudflare.com" in line:
        for word in line.split():
            if "trycloudflare.com" in word and "https://" in word:
                tunnel_url = word.strip()
                print("\n" + "=" * 65)
                print("🎉 GPU WORKER ĐÃ SẴN SÀNG KẾT NỐI VỚI DUBBINGSTATION!")
                print(f"👉 URL CỦA BẠN: {tunnel_url}")
                print("=" * 65)
                print("Copy dòng dưới đây vào file .env của dự án DubbingStation:")
                print(f'VOICE_WORKER_URL="{tunnel_url}"')
                print("=" * 65)
                break